In [0]:
# =============================================================================
# QUESTION 1: TOP 80% GMV GENERATING SUPPLIERS - SUPPLIER-LEVEL ADOPTION
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as patches

# =============================================================================
# METHODOLOGY NOTE
# =============================================================================
print("\n" + "="*100)
print("METHODOLOGY: SUPPLIER-LEVEL FEATURE ADOPTION")
print("="*100)
print("""
A supplier is counted as ADOPTING a feature if they use it on AT LEAST ONE tour.

This means:
- A supplier with 10 tours using "1 ticket category" → Counts as adopter
- A supplier with 1 tour using ">1 ticket category" → Also counts as adopter
- A supplier can be counted in MULTIPLE sub-categories

Example: Supplier with 100 tours
  - 95 tours with 1 ticket category
  - 5 tours with 3 ticket categories
  → Counted in BOTH "1 category" AND ">1 category"

This measures CAPABILITY (what suppliers CAN do), not exclusive behavior.
Sub-category percentages will NOT add up to parent category percentage.
""")
print("="*100)

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# THE 14 FEATURES (CALCULATED AT TOUR LEVEL)
# =============================================================================
# Note: These are calculated at TOUR level, then aggregated with MAX to supplier level
# This captures "does the supplier use this feature on ANY tour?"

df["Multiple tour options >1"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with 1 ticket category"] = (df.get("has_individual_pricing", 0) & ~df.get("has_individual_scales", 0).astype(bool)).astype(int)
df["Group pricing with 1 pricing tier"] = (df.get("has_group_pricing", 0) & ~df.get("has_group_scales", 0).astype(bool)).astype(int)
df["Add-ons with 1 pricing tier"] = (df.get("has_addons", 0) & ~df.get("has_addon_scales", 0).astype(bool)).astype(int)

FEATURES = [
    ("Individual pricing", "has_individual_pricing"),
    ("Individual pricing with 1 ticket category", "Individual pricing with 1 ticket category"),
    ("Individual pricing with >1 ticket category", "has_individual_scales"),
    ("Group pricing", "has_group_pricing"),
    ("Group pricing with 1 pricing tier", "Group pricing with 1 pricing tier"),
    ("Group pricing with >1 pricing tier", "has_group_scales"),
    ("Add-ons", "has_addons"),
    ("Add-ons with 1 pricing tier", "Add-ons with 1 pricing tier"),
    ("Add-ons with >1 pricing tiers", "has_addon_scales"),
    ("Multiple tour options >1", "Multiple tour options >1"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

print(f"\n✓ Using {len(FEATURES)} features for analysis")

# =============================================================================
# AGGREGATE TO SUPPLIER LEVEL FIRST
# =============================================================================
# Using MAX: If ANY tour has the feature, supplier counts as adopter
feature_cols = [col for _, col in FEATURES]

supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg(dict([(col, "max") for col in feature_cols] + [("gmv_l12mo", "sum")]))
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

print("\n" + "="*100)
print("QUESTION 1: FEATURE ADOPTION AMONG TOP 80% GMV SUPPLIERS")
print("="*100)

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS (CUMULATIVE)
# =============================================================================

active_suppliers = supplier_data[supplier_data["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            # Sort by GMV descending
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            
            # Calculate cumulative GMV percentage
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            
            # Mark top 80%
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)

print("\nTop 80% GMV Suppliers by Segment:")
print(f"{'Segment':<20} {'Total Suppliers':<20} {'Top 80% GMV':<20} {'% of Suppliers':<20} {'GMV':<20}")
print("-" * 100)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    seg_data = active_suppliers[active_suppliers["supplier_segment"] == segment]
    total = len(seg_data)
    top_80 = seg_data["is_top_80"].sum()
    pct = 100 * top_80 / total
    gmv = seg_data[seg_data["is_top_80"] == 1]["supplier_gmv_l12mo"].sum() / 1_000_000
    
    print(f"{segment:<20} {total:<20,} {top_80:<20,} {pct:<20.0f}% €{gmv:>18,.1f}M")

# =============================================================================
# CALCULATE ADOPTION RATES (SUPPLIER-LEVEL)
# =============================================================================

segments = sorted(active_suppliers["supplier_segment"].dropna().unique())

top_80_adoption = []
for segment in segments:
    seg_top = active_suppliers[(active_suppliers["supplier_segment"] == segment) & 
                                (active_suppliers["is_top_80"] == 1)]
    
    if len(seg_top) == 0:
        continue
    
    row = {
        "segment": segment,
        "suppliers": len(seg_top),
        "total_gmv": seg_top["supplier_gmv_l12mo"].sum(),
    }
    
    # Calculate % of SUPPLIERS (not tours) that have each feature
    for feat_name, col in FEATURES:
        row[feat_name] = 100.0 * seg_top[col].mean()
    
    top_80_adoption.append(row)

top_80_df = pd.DataFrame(top_80_adoption)
top_80_df = top_80_df.sort_values("total_gmv", ascending=False)

print("\n" + "="*100)
print("FEATURE ADOPTION RATES (% OF SUPPLIERS)")
print("="*100)
print("Note: Sub-categories overlap. A supplier using a feature on ANY tour counts as adopter.\n")

for _, row in top_80_df.iterrows():
    print(f"\n{row['segment']} ({int(row['suppliers'])} suppliers | €{row['total_gmv']/1e6:.1f}M):")
    print(f"{'Feature':<60} {'Adoption %':<15}")
    print("-" * 75)
    
    feature_adoptions = [(feat_name, row[feat_name]) for feat_name, _ in FEATURES]
    feature_adoptions = sorted(feature_adoptions, key=lambda x: x[1], reverse=True)
    
    for feat, adoption in feature_adoptions:
        print(f"{feat:<60} {adoption:>12.1f}%")

# =============================================================================
# SHOW OVERLAP EXAMPLES
# =============================================================================
print("\n" + "="*100)
print("OVERLAP EXPLANATION: Why sub-categories don't add up")
print("="*100)

for _, row in top_80_df.iterrows():
    segment = row['segment']
    
    # Individual pricing
    total_ind = row["Individual pricing"]
    ind_1 = row["Individual pricing with 1 ticket category"]
    ind_multi = row["Individual pricing with >1 ticket category"]
    overlap_ind = ind_1 + ind_multi - total_ind
    
    print(f"\n{segment} - Individual Pricing:")
    print(f"  Total using individual pricing: {total_ind:.1f}%")
    print(f"  Have at least 1 tour with 1 category: {ind_1:.1f}%")
    print(f"  Have at least 1 tour with >1 category: {ind_multi:.1f}%")
    print(f"  → Overlap (use BOTH): {overlap_ind:.1f}% of suppliers")
    print(f"  → This means {overlap_ind:.1f}% have some tours with 1 category AND some tours with >1 category")

print("\n" + "="*100)

# =============================================================================
# CHART 1: HEATMAP BY SEGMENT
# =============================================================================

fig, ax = plt.subplots(figsize=(18, 10))

feature_names = [f[0] for f in FEATURES]
segment_list = top_80_df["segment"].tolist()

matrix = []
for segment in segment_list:
    row = []
    seg_data = top_80_df[top_80_df["segment"] == segment]
    for feat_name, _ in FEATURES:
        row.append(seg_data[feat_name].iloc[0])
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Features
ax.set_xticks(np.arange(len(feature_names)))
ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=9, fontweight="bold")

# Y-axis: Segments
segment_labels = []
for segment in segment_list:
    seg_data = top_80_df[top_80_df["segment"] == segment]
    suppliers = int(seg_data["suppliers"].iloc[0])
    gmv_m = seg_data["total_gmv"].iloc[0] / 1_000_000
    
    # Calculate what % of suppliers this represents
    total_in_segment = active_suppliers[active_suppliers["supplier_segment"] == segment].shape[0]
    pct_suppliers = 100 * suppliers / total_in_segment if total_in_segment > 0 else 0
    
    segment_labels.append(f"{segment}\n({suppliers} suppliers = {pct_suppliers:.0f}% | €{gmv_m:.1f}M = 80% GMV)")

ax.set_yticks(np.arange(len(segment_list)))
ax.set_yticklabels(segment_labels, fontsize=10)

# Title
ax.set_title(
    "What Do Revenue-Driving Suppliers Actually Use?\n"
    "Feature Adoption Rates for Top 80% GMV-Generating Suppliers by Segment\n"
    "Methodology: Supplier counted if feature used on ANY tour (sub-categories may overlap)",
    fontsize=14,
    fontweight="bold",
    pad=25
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=9, 
               fontweight="bold")

# Highlight Leisure Brand API pricing
api_pricing_idx = [i for i, (name, _) in enumerate(FEATURES) if name == "API pricing"][0]
leisure_brand_idx = segment_list.index("Leisure Brand") if "Leisure Brand" in segment_list else None

if leisure_brand_idx is not None:
    rect = patches.Rectangle(
        (api_pricing_idx - 0.48, leisure_brand_idx - 0.48),
        0.96, 0.96,
        linewidth=3,
        edgecolor=ORANGE_DARK,
        facecolor='none',
        linestyle='--'
    )
    ax.add_patch(rect)
    
    leisure_api_adoption = matrix[leisure_brand_idx, api_pricing_idx]
    ax.annotate(
        f'Leisure Brand: {leisure_api_adoption:.0f}% API pricing\n(highest across all segments)',
        xy=(api_pricing_idx, leisure_brand_idx),
        xytext=(api_pricing_idx + 2.5, leisure_brand_idx - 1.5),
        fontsize=10,
        fontweight='bold',
        color=ORANGE_DARK,
        bbox=dict(boxstyle="round,pad=0.5", facecolor=ORANGE_PALE, edgecolor=ORANGE_DARK, linewidth=2, alpha=0.9),
        arrowprops=dict(arrowstyle='->', connectionstyle='arc3,rad=0.3', color=ORANGE_DARK, lw=2.5)
    )

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("% of Suppliers Using Feature on ≥1 Tour", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segment_list)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 1: Feature adoption by segment (supplier-level, {len(FEATURES)} features)")

# =============================================================================
# CHART 2: OVERALL FEATURE RANKING (FIXED LABEL POSITIONING)
# =============================================================================

total_gmv = top_80_df["total_gmv"].sum()
feature_rankings = []

for feat_name, _ in FEATURES:
    weighted_avg = 0
    for _, row in top_80_df.iterrows():
        weight = row["total_gmv"] / total_gmv
        weighted_avg += row[feat_name] * weight
    
    min_val = top_80_df[feat_name].min()
    max_val = top_80_df[feat_name].max()
    
    feature_rankings.append({
        "feature": feat_name,
        "avg_adoption": weighted_avg,
        "min": min_val,
        "max": max_val,
        "range": max_val - min_val
    })

ranking_df = pd.DataFrame(feature_rankings).sort_values("avg_adoption", ascending=True)

fig, ax = plt.subplots(figsize=(14, 10))

y_pos = np.arange(len(ranking_df))

def get_bar_color(adoption_rate):
    if adoption_rate >= 80:
        return ORANGE_DARK
    elif adoption_rate >= 50:
        return ORANGE
    elif adoption_rate >= 20:
        return ORANGE_LIGHT
    else:
        return ORANGE_PALE

bar_colors = [get_bar_color(row["avg_adoption"]) for _, row in ranking_df.iterrows()]
bars = ax.barh(y_pos, ranking_df["avg_adoption"], 
               color=bar_colors, edgecolor=GREY_DARK, linewidth=1.5, alpha=0.9)

# Add range indicators FIRST (so they're in the background)
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    ax.plot([row["min"], row["max"]], [i, i], 
            color=GREY_DARK, linewidth=3, alpha=0.6, zorder=1)
    ax.scatter([row["min"], row["max"]], [i, i], 
               color=GREY_DARK, s=50, zorder=2, alpha=0.8)

# SMART VALUE LABELS - positioned to avoid range lines
THRESHOLD = 70  # If bar > 70%, put label inside
for i, (idx, row) in enumerate(ranking_df.iterrows()):
    value = row["avg_adoption"]
    
    if value > THRESHOLD:
        # Long bar: place label inside on the left, white text, ABOVE center
        ax.text(value - 3, i + 0.25, f"{value:.0f}%", 
               va="center", ha="right", fontsize=9, fontweight="bold", color="white",
               bbox=dict(boxstyle="round,pad=0.3", facecolor='none', edgecolor='none'))
    else:
        # Short bar: place label outside on the right, dark text, with white background
        ax.text(value + 2, i, f"{value:.0f}%", 
               va="center", ha="left", fontsize=9, fontweight="bold", color=GREY_DARK,
               bbox=dict(boxstyle="round,pad=0.3", facecolor='white', edgecolor='none', alpha=0.9))

ax.set_yticks(y_pos)
ax.set_yticklabels(ranking_df["feature"], fontsize=10)
ax.set_xlabel("% of Suppliers Using Feature on ≥1 Tour", fontsize=12, fontweight="bold")
ax.set_title(
    "Feature Adoption Ranking: What Revenue-Driving Suppliers Use Most\n"
    "GMV-weighted average across all segments (Top 80% GMV suppliers only)\n"
    "Bar = weighted average | Line = range across segments | Methodology: Counted if used on ANY tour",
    fontsize=13,
    fontweight="bold",
    pad=20
)
ax.set_xlim(0, 105)
ax.grid(axis="x", alpha=0.3, linestyle="--")

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor=ORANGE_DARK, edgecolor=GREY_DARK, label='High (≥80%)'),
    Patch(facecolor=ORANGE, edgecolor=GREY_DARK, label='Medium-high (50-79%)'),
    Patch(facecolor=ORANGE_LIGHT, edgecolor=GREY_DARK, label='Medium (20-49%)'),
    Patch(facecolor=ORANGE_PALE, edgecolor=GREY_DARK, label='Low (<20%)')
]
ax.legend(handles=legend_elements, loc='lower right', fontsize=10, framealpha=0.95)

# Summary box
top_3 = ranking_df.nlargest(3, "avg_adoption")["feature"].tolist()
summary = "Most Adopted by Revenue Drivers:\n" + "\n".join([f"{i+1}. {f}" for i, f in enumerate(top_3)])
ax.text(
    0.98, 0.28, summary,
    transform=ax.transAxes,
    fontsize=10,
    verticalalignment="bottom",
    horizontalalignment="right",
    bbox=dict(boxstyle="round", facecolor=ORANGE_PALE, alpha=0.4, edgecolor=ORANGE_DARK, linewidth=2)
)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 2: Overall feature ranking (supplier-level, {len(FEATURES)} features)")


In [0]:
# =============================================================================
# QUESTION 2: Managed vs Not Managed - SUPPLIER-LEVEL ADOPTION
# Size buckets based on 33rd and 67th percentiles
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

# =============================================================================
# METHODOLOGY NOTE
# =============================================================================
print("\n" + "="*100)
print("QUESTION 2: HOW DOES MANAGED SUPPORT RELATE TO FEATURE ADOPTION?")
print("="*100)
print("""
METHOD: Supplier-level analysis across all active suppliers (GMV > 0).

KEY DIFFERENCE FROM QUESTION 1:
- Question 1: Top 80% GMV suppliers only (focus on high performers)
- Question 2: ALL active suppliers (to see full managed vs not managed picture across all sizes)

This means adoption rates in Question 2 will be LOWER than Question 1 because:
- Question 1 filtered to top revenue generators who likely use more features
- Question 2 includes the full supplier base, including smaller/newer suppliers

SIZE BUCKETS: Based on GMV tertiles (33rd and 67th percentiles)
- Small: Bottom 33% by GMV
- Medium: Middle 33% by GMV  
- Large: Top 33% by GMV

QUESTION: At the same supplier size, do managed suppliers adopt different features?
""")
print("="*100)

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# is_managed is "yes"/"no"
if "is_managed" in df.columns:
    df["is_managed"] = df["is_managed"].astype(str).str.lower().fillna("no")

# =============================================================================
# ALL 14 FEATURES (SAME AS QUESTION 1)
# =============================================================================
df["Multiple tour options >1"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)
df["Individual pricing with 1 ticket category"] = (df.get("has_individual_pricing", 0) & ~df.get("has_individual_scales", 0).astype(bool)).astype(int)
df["Group pricing with 1 pricing tier"] = (df.get("has_group_pricing", 0) & ~df.get("has_group_scales", 0).astype(bool)).astype(int)
df["Add-ons with 1 pricing tier"] = (df.get("has_addons", 0) & ~df.get("has_addon_scales", 0).astype(bool)).astype(int)

FEATURES = [
    ("Individual pricing", "has_individual_pricing"),
    ("Individual pricing with 1 ticket category", "Individual pricing with 1 ticket category"),
    ("Individual pricing with >1 ticket category", "has_individual_scales"),
    ("Group pricing", "has_group_pricing"),
    ("Group pricing with 1 pricing tier", "Group pricing with 1 pricing tier"),
    ("Group pricing with >1 pricing tier", "has_group_scales"),
    ("Add-ons", "has_addons"),
    ("Add-ons with 1 pricing tier", "Add-ons with 1 pricing tier"),
    ("Add-ons with >1 pricing tiers", "has_addon_scales"),
    ("Multiple tour options >1", "Multiple tour options >1"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

for name, col in FEATURES:
    if col not in df.columns:
        df[col] = 0

print(f"\n✓ Using all {len(FEATURES)} features (consistent with Question 1)")

# =============================================================================
# AGGREGATE TO SUPPLIER LEVEL
# =============================================================================
print("\n" + "="*100)
print("AGGREGATING TO SUPPLIER LEVEL")
print("="*100)

feature_cols = [col for _, col in FEATURES]

# Get supplier-level feature adoption (MAX across all their tours)
# If a supplier uses a feature on ANY tour, they count as adopting it
supplier_features = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg(dict([(col, "max") for col in feature_cols] + [
          ("gmv_l12mo", "sum"),
          ("is_managed", "first")
      ]))
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

print(f"\n✓ Aggregated {len(df):,} tours into {len(supplier_features):,} suppliers")
print(f"  - Each supplier marked as adopting a feature if they use it on ANY tour")
print(f"  - GMV summed across all supplier's tours")

# =============================================================================
# CREATE SIZE BUCKETS BASED ON TERTILES (33rd and 67th percentiles)
# =============================================================================

# Filter to suppliers with GMV > 0
active_suppliers = supplier_features[supplier_features["supplier_gmv_l12mo"] > 0].copy()

print(f"\n✓ Analyzing {len(active_suppliers):,} active suppliers (GMV > 0)")
print(f"  Total GMV: €{active_suppliers['supplier_gmv_l12mo'].sum() / 1_000_000:,.1f}M")

# Calculate tertiles
p33 = active_suppliers["supplier_gmv_l12mo"].quantile(0.33)
p67 = active_suppliers["supplier_gmv_l12mo"].quantile(0.67)

print("\n" + "="*100)
print("SIZE BUCKET DEFINITIONS (Based on GMV Tertiles)")
print("="*100)
print(f"Small:  €0 - €{p33:,.0f} (bottom 33%)")
print(f"Medium: €{p33:,.0f} - €{p67:,.0f} (middle 33%)")
print(f"Large:  €{p67:,.0f}+ (top 33%)")
print("="*100)

def assign_size_bucket_tertile(gmv):
    """Assign supplier to size bucket based on tertiles"""
    if gmv <= 0:
        return "No GMV"
    elif gmv <= p33:
        return "Small"
    elif gmv <= p67:
        return "Medium"
    else:
        return "Large"

active_suppliers["size_bucket"] = active_suppliers["supplier_gmv_l12mo"].apply(assign_size_bucket_tertile)

SIZE_BUCKET_ORDER = ["Small", "Medium", "Large"]

# Show distribution by size and managed status
print("\nSupplier Distribution by Size Bucket and Managed Status:")
print(f"{'Size Bucket':<15} {'Managed':<15} {'Not Managed':<15} {'Total':<15} {'% Managed':<15}")
print("-" * 75)

for bucket in SIZE_BUCKET_ORDER:
    bucket_suppliers = active_suppliers[active_suppliers["size_bucket"] == bucket]
    managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "yes"])
    not_managed = len(bucket_suppliers[bucket_suppliers["is_managed"] == "no"])
    total = managed + not_managed
    pct_managed = 100 * managed / total if total > 0 else 0
    
    print(f"{bucket:<15} {managed:<15,} {not_managed:<15,} {total:<15,} {pct_managed:<15.1f}%")

total_managed = len(active_suppliers[active_suppliers["is_managed"] == "yes"])
total_not_managed = len(active_suppliers[active_suppliers["is_managed"] == "no"])
print("-" * 75)
print(f"{'TOTAL':<15} {total_managed:<15,} {total_not_managed:<15,} {total_managed + total_not_managed:<15,}")

# =============================================================================
# CALCULATE ADOPTION BY SIZE BUCKET AND MANAGED STATUS (SUPPLIER-LEVEL)
# =============================================================================
adoption_data = []

for bucket in SIZE_BUCKET_ORDER:
    # Get suppliers in this size bucket
    bucket_suppliers = active_suppliers[active_suppliers["size_bucket"] == bucket]
    
    managed = bucket_suppliers[bucket_suppliers["is_managed"] == "yes"]
    not_managed = bucket_suppliers[bucket_suppliers["is_managed"] == "no"]
    
    for status_data, status_label in [
        (managed, "Managed"),
        (not_managed, "Not Managed")
    ]:
        if len(status_data) == 0:
            continue
            
        row = {
            "size_bucket": bucket,
            "management_status": status_label,
            "suppliers": len(status_data),
            "gmv": status_data["supplier_gmv_l12mo"].sum(),
        }
        
        # Calculate % of SUPPLIERS (not tours) that have each feature
        for feat_name, col in FEATURES:
            row[feat_name] = 100.0 * status_data[col].mean()
        
        adoption_data.append(row)

adoption_df = pd.DataFrame(adoption_data)

print("\n" + "="*100)
print("SUPPLIER-LEVEL FEATURE ADOPTION RATES")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    print(f"\n{bucket} Suppliers:")
    print(f"{'Feature':<50} {'Managed':<15} {'Not Managed':<15} {'Difference':<15}")
    print("-" * 95)
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    if len(managed_data) == 0 or len(not_managed_data) == 0:
        print("  Insufficient data")
        continue
    
    for feat_name, _ in FEATURES:
        m_val = float(managed_data[feat_name].iloc[0])
        nm_val = float(not_managed_data[feat_name].iloc[0])
        diff = m_val - nm_val
        
        print(f"{feat_name:<50} {m_val:>12.1f}% {nm_val:>12.1f}% {diff:>12.1f}pp")

# =============================================================================
# CHART 1: GROUPED BAR CHART - Feature Adoption by Size Bucket
# Shows Managed vs Not Managed side-by-side for each size bucket
# =============================================================================

fig, axes = plt.subplots(1, 3, figsize=(28, 10), sharey=True)

feature_names = [f[0] for f in FEATURES]

for idx, bucket in enumerate(SIZE_BUCKET_ORDER):
    ax = axes[idx]
    
    bucket_data = adoption_df[adoption_df["size_bucket"] == bucket]
    managed_data = bucket_data[bucket_data["management_status"] == "Managed"]
    not_managed_data = bucket_data[bucket_data["management_status"] == "Not Managed"]
    
    x = np.arange(len(feature_names))
    width = 0.35
    
    managed_values = [float(managed_data[feat].iloc[0]) if len(managed_data) > 0 else 0 for feat in feature_names]
    not_managed_values = [float(not_managed_data[feat].iloc[0]) if len(not_managed_data) > 0 else 0 for feat in feature_names]
    
    bars1 = ax.bar(x - width/2, managed_values, width, label='Managed', 
                   color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    bars2 = ax.bar(x + width/2, not_managed_values, width, label='Not Managed', 
                   color=GREY_LIGHT, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    # Add value labels on bars (only if >5%)
    for bars in [bars1, bars2]:
        for bar in bars:
            height = bar.get_height()
            if height > 5:
                ax.text(bar.get_x() + bar.get_width()/2., height + 2,
                       f'{height:.0f}%',
                       ha='center', va='bottom', fontsize=7, fontweight='bold', color=GREY_DARK)
    
    # Get supplier counts
    m_suppliers = int(managed_data["suppliers"].iloc[0]) if len(managed_data) > 0 else 0
    nm_suppliers = int(not_managed_data["suppliers"].iloc[0]) if len(not_managed_data) > 0 else 0
    
    # Get GMV range for this bucket
    bucket_suppliers = active_suppliers[active_suppliers["size_bucket"] == bucket]
    if len(bucket_suppliers) > 0:
        min_gmv = bucket_suppliers["supplier_gmv_l12mo"].min()
        max_gmv = bucket_suppliers["supplier_gmv_l12mo"].max()
        gmv_range = f"€{min_gmv:,.0f} to €{max_gmv:,.0f}"
    else:
        gmv_range = "N/A"
    
    ax.set_title(f'{bucket}\n{gmv_range}\n{m_suppliers} Managed | {nm_suppliers} Not Managed', 
                fontsize=11, fontweight='bold')
    ax.set_xticks(x)
    ax.set_xticklabels(feature_names, rotation=45, ha='right', fontsize=8)
    ax.set_ylim(0, 100)
    ax.grid(axis='y', alpha=0.3, linestyle='--')
    
    if idx == 0:
        ax.set_ylabel('Supplier Adoption Rate (%)', fontsize=12, fontweight='bold')
        ax.legend(fontsize=10, loc='upper left')

plt.suptitle(
    'Managed vs Not Managed: Supplier-Level Feature Adoption Across Size Buckets\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV | All active suppliers included\n'
    'Question: At the same supplier size, does managed support relate to different adoption patterns?',
    fontsize=14,
    fontweight='bold',
    y=0.98
)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 1: Supplier-level feature adoption by size bucket (all {len(FEATURES)} features)")

# =============================================================================
# CHART 2: HEATMAP - Adoption Rates by Size x Management Status
# Rows = Features, Columns = Size buckets split by Managed/Not Managed
# =============================================================================

fig, ax = plt.subplots(figsize=(14, 14))

# Build matrix: features (rows) x (size_bucket × management_status) (columns)
column_labels = []
matrix = []

for feat_name, _ in FEATURES:
    row = []
    for bucket in SIZE_BUCKET_ORDER:
        for status in ["Managed", "Not Managed"]:
            data = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                              (adoption_df["management_status"] == status)]
            
            if len(data) > 0:
                row.append(data[feat_name].iloc[0])
            else:
                row.append(0)
            
            # Build column labels (only once)
            if len(matrix) == 0:
                supplier_count = int(data["suppliers"].iloc[0]) if len(data) > 0 else 0
                short_status = "M" if status == "Managed" else "NM"
                column_labels.append(f"{bucket}\n{short_status}\n({supplier_count})")
    
    matrix.append(row)

matrix = np.array(matrix)

# Create colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)

# Heatmap
im = ax.imshow(matrix, aspect="auto", cmap=cmap, vmin=0, vmax=100)

# X-axis: Size × Management combinations
ax.set_xticks(np.arange(len(column_labels)))
ax.set_xticklabels(column_labels, rotation=45, ha='right', fontsize=10)

# Y-axis: Features
ax.set_yticks(np.arange(len(feature_names)))
ax.set_yticklabels(feature_names, fontsize=10, fontweight='bold')

# Title
ax.set_title(
    'Supplier-Level Feature Adoption: Managed vs Not Managed Across Supplier Sizes\n'
    'Size buckets: Small (bottom 33%), Medium (middle 33%), Large (top 33%) by GMV | All active suppliers\n'
    'Question: Does managed support relate to adoption when controlling for supplier size?',
    fontsize=13,
    fontweight='bold',
    pad=20
)

# Annotate cells
for i in range(matrix.shape[0]):
    for j in range(matrix.shape[1]):
        val = matrix[i, j]
        text_color = "white" if val > 55 else GREY_DARK
        ax.text(j, i, f"{val:.0f}%", 
               ha="center", va="center", 
               color=text_color, 
               fontsize=9, 
               fontweight="bold")

# Add vertical lines to separate size buckets
for i in range(1, len(SIZE_BUCKET_ORDER)):
    ax.axvline(i * 2 - 0.5, color=GREY_DARK, linewidth=3, alpha=0.7)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Supplier Adoption Rate (%)", fontsize=12, fontweight="bold")
cbar.ax.tick_params(labelsize=11)

# Gridlines
ax.set_xticks(np.arange(len(column_labels)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(feature_names)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=2)
ax.tick_params(which="minor", size=0)

plt.tight_layout()
plt.show()

print(f"\n✓ Chart 2: Heatmap showing supplier-level adoption across size buckets (all {len(FEATURES)} features)")

# =============================================================================
# SUMMARY: WHERE DOES MANAGED SUPPORT MATTER?
# =============================================================================
print("\n" + "="*100)
print("SUMMARY: Features Where Managed Support Shows Different Adoption (by size bucket)")
print("="*100)

for bucket in SIZE_BUCKET_ORDER:
    bucket_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                  (adoption_df["management_status"] == "Managed")]
    bucket_not_managed = adoption_df[(adoption_df["size_bucket"] == bucket) & 
                                      (adoption_df["management_status"] == "Not Managed")]
    
    if len(bucket_managed) == 0 or len(bucket_not_managed) == 0:
        continue
    
    print(f"\n{bucket}:")
    
    differences = []
    for feat_name, _ in FEATURES:
        m_val = float(bucket_managed[feat_name].iloc[0])
        nm_val = float(bucket_not_managed[feat_name].iloc[0])
        diff = m_val - nm_val
        
        if abs(diff) > 5:  # Show differences > 5pp
            differences.append((feat_name, m_val, nm_val, diff))
    
    if differences:
        differences = sorted(differences, key=lambda x: abs(x[3]), reverse=True)
        for feat, m_val, nm_val, diff in differences:
            direction = "higher" if diff > 0 else "lower"
            
            # Mark as significant if >10pp
            if abs(diff) > 10:
                marker = "***"
            else:
                marker = ""
            
            print(f"  {feat}: Managed {m_val:.1f}% vs Not Managed {nm_val:.1f}% ({diff:+.1f}pp {direction}) {marker}")
    else:
        print(f"  No meaningful differences (all features <5pp gap)")

# =============================================================================
# KEY INSIGHTS
# =============================================================================
print("\n" + "="*100)
print("KEY INSIGHTS")
print("="*100)

print("\n1. SIZE EFFECT vs SUPPORT EFFECT:")
print("   Comparing adoption across Small → Medium → Large buckets vs Managed/Not Managed")

# Calculate size effect (average increase from Small to Large)
size_effect_data = []
for feat_name, _ in FEATURES:
    small_avg = adoption_df[adoption_df["size_bucket"] == "Small"][feat_name].mean()
    large_avg = adoption_df[adoption_df["size_bucket"] == "Large"][feat_name].mean()
    size_effect = large_avg - small_avg
    
    # Calculate support effect (average across all sizes)
    managed_avg = adoption_df[adoption_df["management_status"] == "Managed"][feat_name].mean()
    not_managed_avg = adoption_df[adoption_df["management_status"] == "Not Managed"][feat_name].mean()
    support_effect = managed_avg - not_managed_avg
    
    size_effect_data.append({
        "feature": feat_name,
        "size_effect": size_effect,
        "support_effect": support_effect
    })

size_effect_df = pd.DataFrame(size_effect_data).sort_values("size_effect", ascending=False)

print(f"\n   {'Feature':<50} {'Size Effect':<15} {'Support Effect':<15} {'Dominant Factor':<15}")
print("   " + "-" * 95)

for _, row in size_effect_df.iterrows():
    size_eff = row["size_effect"]
    support_eff = row["support_effect"]
    
    if abs(size_eff) > abs(support_eff) * 2:
        dominant = "SIZE"
    elif abs(support_eff) > abs(size_eff) * 2:
        dominant = "SUPPORT"
    else:
        dominant = "BOTH"
    
    print(f"   {row['feature']:<50} {size_eff:>12.1f}pp {support_eff:>12.1f}pp {dominant:>15}")

print("\n2. FEATURES WHERE MANAGED SUPPORT SHOWS STRONGEST RELATIONSHIP:")
features_where_support_matters = size_effect_df[size_effect_df["support_effect"].abs() > 5].sort_values("support_effect", ascending=False)
if len(features_where_support_matters) > 0:
    for _, row in features_where_support_matters.head(5).iterrows():
        print(f"   - {row['feature']}: {row['support_effect']:+.1f}pp for managed")
else:
    print("   - No features show >5pp managed support difference")

print("\n3. FEATURES WHERE SIZE SHOWS STRONGEST RELATIONSHIP:")
features_where_size_matters = size_effect_df[size_effect_df["size_effect"].abs() > 10].sort_values("size_effect", ascending=False)
if len(features_where_size_matters) > 0:
    for _, row in features_where_size_matters.head(5).iterrows():
        print(f"   - {row['feature']}: {row['size_effect']:+.1f}pp increase from Small to Large")
else:
    print("   - No features show >10pp size effect")

print("\n" + "="*100)
print(f"✓ Analysis complete: Managed vs Not Managed (SUPPLIER-LEVEL) with all {len(FEATURES)} features")
print(f"✓ Sample: {len(active_suppliers):,} active suppliers")
print(f"✓ Total GMV: €{active_suppliers['supplier_gmv_l12mo'].sum() / 1_000_000:,.1f}M")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 3: Feature Combinations Among Top 80% GMV Suppliers
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# =============================================================================
# METHODOLOGY NOTE
# =============================================================================
print("\n" + "="*100)
print("QUESTION 3: DO TOP GMV SUPPLIERS USE FEATURE COMBINATIONS?")
print("="*100)
print("""
METHOD: Supplier-level analysis of top 80% GMV suppliers (same as Question 1).

QUESTION: Do high-performing suppliers layer multiple pricing features together,
or do they primarily use single features in isolation?

This analysis examines:
1. How many features suppliers use simultaneously
2. Which specific feature combinations appear most frequently
3. Whether combinations are common or rare among top performers
""")
print("="*100)

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
    "has_native_scales", "has_api_scales", "has_scale_pricing",
    "has_static_api_pricing", "has_live_dynamic_pricing", "uses_api_pricing",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

# =============================================================================
# ALL 11 NON-OVERLAPPING FEATURES FOR COMBINATION ANALYSIS
# =============================================================================
df["Multiple tour options >1"] = (pd.to_numeric(df.get("total_options", 0), errors="coerce").fillna(0) > 1).astype(int)

# For combination analysis, use a simplified feature set to avoid overlaps
COMBINATION_FEATURES = [
    ("Individual pricing", "has_individual_pricing"),
    ("Individual pricing with >1 ticket category", "has_individual_scales"),
    ("Group pricing", "has_group_pricing"),
    ("Group pricing with >1 pricing tier", "has_group_scales"),
    ("Add-ons", "has_addons"),
    ("Add-ons with >1 pricing tiers", "has_addon_scales"),
    ("Multiple tour options >1", "Multiple tour options >1"),
    ("Any pricing scales", "has_scale_pricing"),
    ("API pricing", "uses_api_pricing"),
    ("API pricing scales", "has_api_scales"),
    ("Live pricing and availability", "has_live_dynamic_pricing"),
]

for name, col in COMBINATION_FEATURES:
    if col not in df.columns:
        df[col] = 0

print(f"\n✓ Analyzing {len(COMBINATION_FEATURES)} features for combinations")

# =============================================================================
# AGGREGATE TO SUPPLIER LEVEL
# =============================================================================
feature_cols = [col for _, col in COMBINATION_FEATURES]

supplier_data = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg(dict([(col, "max") for col in feature_cols] + [("gmv_l12mo", "sum")]))
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

print(f"\n✓ Aggregated to {len(supplier_data):,} suppliers")

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS (CUMULATIVE)
# =============================================================================

active_suppliers = supplier_data[supplier_data["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)

print("\n" + "="*100)
print("Top 80% GMV Suppliers by Segment:")
print(f"{'Segment':<20} {'Total Suppliers':<20} {'Top 80% GMV':<20} {'%':<15}")
print("-" * 75)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    seg_data = active_suppliers[active_suppliers["supplier_segment"] == segment]
    total = len(seg_data)
    top_80 = seg_data["is_top_80"].sum()
    pct = 100 * top_80 / total
    print(f"{segment:<20} {total:<20,} {top_80:<20,} {pct:<15.1f}%")

# =============================================================================
# CALCULATE FEATURE COUNT AND COMBO STRING (BEFORE FILTERING)
# =============================================================================

def count_features(row):
    count = 0
    for _, feat_col in COMBINATION_FEATURES:
        if row[feat_col] == 1:
            count += 1
    return count

def create_combo_string(row):
    active = []
    for feat_name, feat_col in COMBINATION_FEATURES:
        if row[feat_col] == 1:
            active.append(feat_name)
    
    if len(active) == 0:
        return "No features"
    
    return " + ".join(sorted(active))

active_suppliers["feature_count"] = active_suppliers.apply(count_features, axis=1)
active_suppliers["combo"] = active_suppliers.apply(create_combo_string, axis=1)

# NOW filter to top 80%
top_80 = active_suppliers[active_suppliers["is_top_80"] == 1].copy()

# =============================================================================
# ANALYSIS 1: FEATURE COUNT DISTRIBUTION
# =============================================================================

print("\n" + "="*100)
print("FEATURE COUNT DISTRIBUTION - TOP 80% GMV SUPPLIERS")
print("="*100)

print(f"\nOverall (all segments combined, n={len(top_80):,}):")
print(f"{'Features Used':<20} {'Suppliers':<15} {'%':<15}")
print("-" * 50)

for count in sorted(top_80["feature_count"].unique()):
    count_suppliers = len(top_80[top_80["feature_count"] == count])
    pct = 100 * count_suppliers / len(top_80)
    print(f"{count:<20} {count_suppliers:<15,} {pct:<15.1f}%")

print(f"\nAverage features per supplier: {top_80['feature_count'].mean():.2f}")
print(f"Median features per supplier: {top_80['feature_count'].median():.0f}")

# By segment
print("\n" + "="*100)
print("BY SEGMENT:")
print("="*100)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    seg_top_80 = top_80[top_80["supplier_segment"] == segment]
    
    print(f"\n{segment} (n={len(seg_top_80):,}):")
    print(f"{'Features Used':<20} {'Suppliers':<15} {'%':<15}")
    print("-" * 50)
    
    for count in sorted(seg_top_80["feature_count"].unique()):
        count_suppliers = len(seg_top_80[seg_top_80["feature_count"] == count])
        pct = 100 * count_suppliers / len(seg_top_80)
        print(f"{count:<20} {count_suppliers:<15,} {pct:<15.1f}%")
    
    avg = seg_top_80["feature_count"].mean()
    median = seg_top_80["feature_count"].median()
    print(f"\nAverage: {avg:.2f} | Median: {median:.0f}")

# =============================================================================
# ANALYSIS 2: SINGLE VS MULTIPLE FEATURES
# =============================================================================

print("\n" + "="*100)
print("SINGLE FEATURE VS COMBINATIONS")
print("="*100)

print(f"\n{'Segment':<20} {'0 Features':<15} {'1 Feature':<15} {'2-3 Features':<15} {'4+ Features':<15}")
print("-" * 80)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    seg_top_80 = top_80[top_80["supplier_segment"] == segment]
    
    zero = 100 * len(seg_top_80[seg_top_80["feature_count"] == 0]) / len(seg_top_80) if len(seg_top_80) > 0 else 0
    one = 100 * len(seg_top_80[seg_top_80["feature_count"] == 1]) / len(seg_top_80) if len(seg_top_80) > 0 else 0
    two_three = 100 * len(seg_top_80[(seg_top_80["feature_count"] >= 2) & (seg_top_80["feature_count"] <= 3)]) / len(seg_top_80) if len(seg_top_80) > 0 else 0
    four_plus = 100 * len(seg_top_80[seg_top_80["feature_count"] >= 4]) / len(seg_top_80) if len(seg_top_80) > 0 else 0
    
    print(f"{segment:<20} {zero:<15.1f}% {one:<15.1f}% {two_three:<15.1f}% {four_plus:<15.1f}%")

# Overall
zero = 100 * len(top_80[top_80["feature_count"] == 0]) / len(top_80)
one = 100 * len(top_80[top_80["feature_count"] == 1]) / len(top_80)
two_three = 100 * len(top_80[(top_80["feature_count"] >= 2) & (top_80["feature_count"] <= 3)]) / len(top_80)
four_plus = 100 * len(top_80[top_80["feature_count"] >= 4]) / len(top_80)

print("-" * 80)
print(f"{'OVERALL':<20} {zero:<15.1f}% {one:<15.1f}% {two_three:<15.1f}% {four_plus:<15.1f}%")

# =============================================================================
# TOP 10 COMBINATIONS PER SEGMENT (TOP 80% ONLY)
# =============================================================================

print("\n" + "="*100)
print("TOP 10 FEATURE COMBINATIONS - TOP 80% GMV SUPPLIERS PER SEGMENT")
print("="*100)

for segment in sorted(active_suppliers["supplier_segment"].dropna().unique()):
    seg_top_80 = top_80[top_80["supplier_segment"] == segment]
    
    print(f"\n{'='*120}")
    print(f"{segment} (n={len(seg_top_80):,} suppliers generating 80% of GMV)")
    print(f"{'='*120}")
    
    top_combos = seg_top_80["combo"].value_counts().head(10)
    
    print(f"{'Rank':<6} {'Count':<10} {'%':<10} {'Combination'}")
    print("-" * 120)
    
    for rank, (combo, count) in enumerate(top_combos.items(), 1):
        pct = 100 * count / len(seg_top_80)
        feature_count = len(combo.split(" + ")) if combo != "No features" else 0
        print(f"{rank:<6} {count:<10} {pct:<10.1f} [{feature_count} features] {combo}")

# =============================================================================
# CHART 1: FEATURE COUNT DISTRIBUTION BY SEGMENT
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 8))

segments = sorted(active_suppliers["supplier_segment"].dropna().unique())
x = np.arange(len(segments))
width = 0.15

feature_buckets = [0, 1, 2, 3, 4, 5]
colors = [GREY_PALE, GREY_LIGHT, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK]

for i, bucket in enumerate(feature_buckets):
    values = []
    for segment in segments:
        seg_top_80 = top_80[top_80["supplier_segment"] == segment]
        
        if len(seg_top_80) == 0:
            values.append(0)
            continue
        
        if bucket == 5:
            pct = 100 * len(seg_top_80[seg_top_80["feature_count"] >= 5]) / len(seg_top_80)
        else:
            pct = 100 * len(seg_top_80[seg_top_80["feature_count"] == bucket]) / len(seg_top_80)
        
        values.append(pct)
    
    label = f"{bucket} features" if bucket < 5 else "5+ features"
    ax.bar(x + i*width, values, width, label=label, color=colors[i], 
           edgecolor=GREY_DARK, linewidth=1, alpha=0.9)

ax.set_xlabel('Segment', fontsize=12, fontweight='bold')
ax.set_ylabel('% of Top 80% GMV Suppliers', fontsize=12, fontweight='bold')
ax.set_title(
    'Feature Count Distribution: Do Top Performers Use Combinations?\n'
    'Top 80% GMV suppliers per segment',
    fontsize=14,
    fontweight='bold',
    pad=20
)
ax.set_xticks(x + width * 2.5)
ax.set_xticklabels(segments, rotation=45, ha='right')
ax.legend(loc='upper right', fontsize=10)
ax.grid(axis='y', alpha=0.3, linestyle='--')
ax.set_ylim(0, 100)

plt.tight_layout()
plt.show()

print("\n✓ Chart 1: Feature count distribution")

# =============================================================================
# CHART 2: TOP 10 COMBINATIONS PER SEGMENT
# =============================================================================

fig, axes = plt.subplots(2, 3, figsize=(28, 16))
axes = axes.flatten()

for idx, segment in enumerate(segments):
    if idx >= len(axes):
        break
    
    ax = axes[idx]
    
    seg_top_80 = top_80[top_80["supplier_segment"] == segment]
    
    top_combos = seg_top_80["combo"].value_counts().head(10)
    
    values = []
    labels = []
    
    for combo, count in top_combos.items():
        pct = 100 * count / len(seg_top_80)
        values.append(pct)
        
        feature_count = len(combo.split(" + ")) if combo != "No features" else 0
        
        # Shorten label if too long
        if len(combo) > 60:
            combo_short = combo[:57] + "..."
        else:
            combo_short = combo
        
        labels.append(f"[{feature_count}] {combo_short}")
    
    y = np.arange(len(labels))
    
    bars = ax.barh(y, values, color=ORANGE_DARK, edgecolor=GREY_DARK, linewidth=1.2, alpha=0.9)
    
    for bar in bars:
        width_val = bar.get_width()
        ax.text(width_val + 0.5, bar.get_y() + bar.get_height()/2.,
               f'{width_val:.1f}%',
               ha='left', va='center', fontsize=9, fontweight='bold', color=GREY_DARK)
    
    ax.set_title(f'{segment}\n{len(seg_top_80):,} suppliers (top 80% GMV)', 
                fontsize=12, fontweight='bold')
    ax.set_xlabel('% of Top 80% Suppliers', fontsize=11)
    ax.set_yticks(y)
    ax.set_yticklabels(labels, fontsize=8)
    ax.grid(axis='x', alpha=0.3, linestyle='--')
    if len(values) > 0:
        ax.set_xlim(0, max(values) * 1.15)

for idx in range(len(segments), len(axes)):
    axes[idx].axis('off')

plt.suptitle(
    'Most Common Feature Combinations Among Top Performers\n'
    'Top 80% GMV-generating suppliers per segment | [n] = number of features in combination',
    fontsize=16,
    fontweight='bold',
    y=0.995
)

plt.tight_layout()
plt.show()

print("\n✓ Chart 2: Top combinations per segment")

# =============================================================================
# KEY INSIGHTS
# =============================================================================

print("\n" + "="*100)
print("KEY INSIGHTS: DO TOP PERFORMERS USE COMBINATIONS?")
print("="*100)

single_feature_pct = 100 * len(top_80[top_80["feature_count"] == 1]) / len(top_80)
two_plus_pct = 100 * len(top_80[top_80["feature_count"] >= 2]) / len(top_80)
avg_features = top_80["feature_count"].mean()

print(f"\n1. OVERALL PATTERN:")
print(f"   - {single_feature_pct:.1f}% of top 80% GMV suppliers use only 1 feature")
print(f"   - {two_plus_pct:.1f}% use 2 or more features")
print(f"   - Average: {avg_features:.2f} features per supplier")

print(f"\n2. BY SEGMENT:")
for segment in segments:
    seg_top_80 = top_80[top_80["supplier_segment"] == segment]
    if len(seg_top_80) == 0:
        continue
    single = 100 * len(seg_top_80[seg_top_80["feature_count"] == 1]) / len(seg_top_80)
    avg = seg_top_80["feature_count"].mean()
    print(f"   - {segment}: {single:.1f}% use 1 feature | Avg: {avg:.2f}")

print(f"\n3. MOST COMMON PATTERN OVERALL:")
overall_top = top_80["combo"].value_counts().head(1)
for combo, count in overall_top.items():
    pct = 100 * count / len(top_80)
    feature_count = len(combo.split(" + ")) if combo != "No features" else 0
    print(f"   - {combo}")
    print(f"   - {count:,} suppliers ({pct:.1f}%)")
    print(f"   - {feature_count} feature(s)")

print("\n" + "="*100)
print("✓ Analysis complete: Feature combinations among top 80% GMV suppliers")
print("="*100)


In [0]:
# =============================================================================
# QUESTION 4: CONFIGURATION DEPTH AMONG ADOPTERS
# =============================================================================

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap
import matplotlib.patches as mpatches

# =============================================================================
# METHODOLOGY NOTE
# =============================================================================
print("\n" + "="*100)
print("QUESTION 4: HOW DEEPLY DO ADOPTERS CONFIGURE FEATURES?")
print("="*100)
print("""
METHOD: Among top 80% GMV suppliers who adopt a feature, how many items do they 
configure on average per tour?

FEATURES ANALYZED (features with configurable depth):
1. Multiple tour options >1: Among suppliers with >1 option, how many options per tour?
2. Add-ons: Among suppliers with ≥1 add-on, how many add-ons per tour?
3. Individual pricing with >1 ticket category: Among adopters, how many categories per tour?
4. Group pricing with >1 pricing tier: Among adopters, how many tiers per tour?

CALCULATION: 
1. Identify top 80% GMV suppliers per segment
2. Identify which suppliers adopt each feature (if used on at least one tour)
3. Filter to tours from adopting suppliers
4. Calculate average items per tour across all tours from adopters

This measures whether suppliers who adopt a feature use it shallowly (minimal configuration)
or deeply (extensive configuration).
""")
print("="*100)

# =============================================================================
# STYLE
# =============================================================================
ORANGE_DARK  = "#D35400"
ORANGE       = "#E67E22"
ORANGE_LIGHT = "#F39C12"
ORANGE_PALE  = "#F8B500"
GREY_DARK    = "#2C3E50"
GREY         = "#7F8C8D"
GREY_LIGHT   = "#BDC3C7"
GREY_PALE    = "#ECF0F1"

# =============================================================================
# LOAD DATA
# =============================================================================
df = spark.sql("SELECT * FROM production.supply_analytics.pricing_feature_combined").toPandas()

# Type fixes
NUM_COLS = ["gmv_l12mo", "total_options", "num_addons", "num_individual_categories", 
            "num_individual_tiers", "num_group_tiers"]
for c in NUM_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0)

FLAG_COLS = [
    "has_individual_pricing", "has_group_pricing", "has_addons",
    "has_individual_scales", "has_group_scales", "has_addon_scales",
]
for c in FLAG_COLS:
    if c in df.columns:
        df[c] = pd.to_numeric(df[c], errors="coerce").fillna(0).astype(int)

print(f"\n✓ Loaded {len(df):,} tour records")

# =============================================================================
# IDENTIFY TOP 80% GMV SUPPLIERS
# =============================================================================

# Aggregate to supplier level to identify top 80%
supplier_gmv = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)["gmv_l12mo"]
      .sum()
      .reset_index()
      .rename(columns={"gmv_l12mo": "supplier_gmv_l12mo"})
)

active_suppliers = supplier_gmv[supplier_gmv["supplier_gmv_l12mo"] > 0].copy()

def identify_top_80_percent(df):
    df_copy = df.copy()
    df_copy["is_top_80"] = 0
    
    for segment in df["supplier_segment"].dropna().unique():
        segment_mask = df_copy["supplier_segment"] == segment
        segment_data = df_copy[segment_mask].copy()
        
        if len(segment_data) > 0:
            segment_data = segment_data.sort_values("supplier_gmv_l12mo", ascending=False)
            total_gmv = segment_data["supplier_gmv_l12mo"].sum()
            segment_data["cumulative_gmv"] = segment_data["supplier_gmv_l12mo"].cumsum()
            segment_data["cumulative_pct"] = segment_data["cumulative_gmv"] / total_gmv
            top_80_suppliers = segment_data[segment_data["cumulative_pct"] <= 0.80]["supplier_id"]
            df_copy.loc[df_copy["supplier_id"].isin(top_80_suppliers), "is_top_80"] = 1
    
    return df_copy

active_suppliers = identify_top_80_percent(active_suppliers)
top_80_supplier_ids = set(active_suppliers[active_suppliers["is_top_80"] == 1]["supplier_id"])

print(f"\n✓ Identified {len(top_80_supplier_ids):,} top 80% GMV suppliers")

# Merge back to tour-level data
df["is_top_80"] = df["supplier_id"].isin(top_80_supplier_ids).astype(int)

# =============================================================================
# IDENTIFY ADOPTERS AT SUPPLIER LEVEL
# =============================================================================

# For each supplier, determine if they are an adopter of each feature
supplier_adoption = (
    df.groupby(["supplier_id", "supplier_segment"], dropna=False)
      .agg({
          "total_options": "max",
          "num_addons": "max",
          "num_individual_categories": "max",
          "num_group_tiers": "max",
      })
      .reset_index()
)

# Define adopter flags
supplier_adoption["adopts_multiple_options"] = (supplier_adoption["total_options"] > 1).astype(int)
supplier_adoption["adopts_addons"] = (supplier_adoption["num_addons"] > 0).astype(int)
supplier_adoption["adopts_ind_categories"] = (supplier_adoption["num_individual_categories"] > 1).astype(int)
supplier_adoption["adopts_grp_tiers"] = (supplier_adoption["num_group_tiers"] > 1).astype(int)

# Merge adopter flags back to tour data
df = df.merge(
    supplier_adoption[["supplier_id", "adopts_multiple_options", "adopts_addons", 
                       "adopts_ind_categories", "adopts_grp_tiers"]],
    on="supplier_id",
    how="left"
)

# =============================================================================
# CALCULATE DEPTH: COUNT PER TOUR, THEN AVERAGE ACROSS SUPPLIERS
# =============================================================================

segments = sorted(df[df["is_top_80"] == 1]["supplier_segment"].dropna().unique())

# Define metrics: (Display Name, Count Column, Adopter Flag)
metrics = [
    ("Multiple tour options >1", "total_options", "adopts_multiple_options"),
    ("Add-ons", "num_addons", "adopts_addons"),
    ("Individual pricing with >1 ticket category", "num_individual_categories", "adopts_ind_categories"),
    ("Group pricing with >1 pricing tier", "num_group_tiers", "adopts_grp_tiers"),
]

results = []

print("\n" + "="*100)
print("CONFIGURATION DEPTH BY SEGMENT (ADOPTERS ONLY)")
print("="*100)

for metric_name, count_col, adopter_flag in metrics:
    print(f"\n{metric_name}:")
    
    # Filter to tours from top 80% suppliers who adopt this feature
    adopter_tours = df[(df["is_top_80"] == 1) & (df[adopter_flag] == 1)].copy()
    
    if len(adopter_tours) > 0:
        # Count items per tour, then average across all tours from adopters
        overall_mean = adopter_tours[count_col].mean()
        overall_median = adopter_tours[count_col].median()
        num_adopters = adopter_tours["supplier_id"].nunique()
        num_tours = len(adopter_tours)
        
        print(f"  Platform-wide: {num_adopters:,} adopting suppliers | {num_tours:,} tours")
        print(f"  Overall depth: Mean={overall_mean:.1f}, Median={overall_median:.1f} items per tour")
    else:
        print(f"  Platform-wide: 0 adopters")
    
    for segment in segments:
        # Total suppliers in segment
        seg_suppliers = df[(df["is_top_80"] == 1) & 
                          (df["supplier_segment"] == segment)]["supplier_id"].nunique()
        
        # Tours from adopters in this segment
        seg_adopter_tours = adopter_tours[adopter_tours["supplier_segment"] == segment]
        
        if len(seg_adopter_tours) > 0:
            mean_val = seg_adopter_tours[count_col].mean()
            median_val = seg_adopter_tours[count_col].median()
            num_seg_adopters = seg_adopter_tours["supplier_id"].nunique()
            num_seg_tours = len(seg_adopter_tours)
            
            seg_adoption_rate = 100 * num_seg_adopters / seg_suppliers if seg_suppliers > 0 else 0
            
            print(f"    {segment}: {num_seg_adopters:,} adopters ({seg_adoption_rate:.0f}% of segment) | "
                  f"{num_seg_tours:,} tours | Mean={mean_val:.1f}, Median={median_val:.1f} per tour")
            
            results.append({
                "metric": metric_name,
                "segment": segment,
                "adopters": num_seg_adopters,
                "tours": num_seg_tours,
                "total_segment": seg_suppliers,
                "adoption_pct": seg_adoption_rate,
                "mean": mean_val,
                "median": median_val
            })
        else:
            results.append({
                "metric": metric_name,
                "segment": segment,
                "adopters": 0,
                "tours": 0,
                "total_segment": seg_suppliers,
                "adoption_pct": 0,
                "mean": np.nan,
                "median": np.nan
            })

results_df = pd.DataFrame(results)

# =============================================================================
# CHART: CONFIGURATION DEPTH HEATMAP
# =============================================================================

fig, ax = plt.subplots(figsize=(16, 10))

metric_names = [m[0] for m in metrics]

# Build depth matrix
depth_matrix = []
segment_labels = []

for segment in segments:
    row = []
    seg_results = results_df[results_df["segment"] == segment]
    
    # Build label with supplier count context
    if len(seg_results) > 0:
        total_in_seg = seg_results["total_segment"].iloc[0]
        segment_labels.append(f"{segment}\n(n={total_in_seg:,})")
    else:
        segment_labels.append(f"{segment}\n(n=0)")
    
    for metric_name in metric_names:
        metric_data = seg_results[seg_results["metric"] == metric_name]
        if len(metric_data) > 0 and metric_data["adopters"].iloc[0] > 0:
            row.append(metric_data["mean"].iloc[0])
        else:
            row.append(np.nan)
    
    depth_matrix.append(row)

depth_matrix = np.array(depth_matrix)
masked_depth = np.ma.masked_invalid(depth_matrix)

# Colormap
cmap = LinearSegmentedColormap.from_list(
    "grey_to_orange",
    [GREY_PALE, ORANGE_PALE, ORANGE_LIGHT, ORANGE, ORANGE_DARK],
    N=256
)
cmap.set_bad(color='white')

# Get valid range for colorbar
valid_values = depth_matrix[~np.isnan(depth_matrix)]
if len(valid_values) > 0:
    vmin, vmax = valid_values.min(), valid_values.max()
else:
    vmin, vmax = 0, 1

im = ax.imshow(masked_depth, aspect="auto", cmap=cmap, vmin=vmin, vmax=vmax)

# Axes
ax.set_xticks(np.arange(len(metric_names)))
ax.set_xticklabels([name.replace(" with ", "\nwith ") for name in metric_names], 
                    fontsize=11, fontweight='bold')
ax.set_yticks(np.arange(len(segments)))
ax.set_yticklabels(segment_labels, fontsize=11)

# Title
ax.set_title(
    'Configuration Depth Among Adopters: Average Items Per Tour\n'
    'Top 80% GMV suppliers only | Question: Do suppliers use features shallowly or deeply?',
    fontsize=15,
    fontweight='bold',
    pad=20
)

# Annotate cells with mean values and adopter counts
for i in range(len(segments)):
    for j in range(len(metric_names)):
        val = depth_matrix[i, j]
        
        if not np.isnan(val):
            # Get adopter count for this cell
            seg = segments[i]
            metric = metric_names[j]
            cell_data = results_df[(results_df["segment"] == seg) & 
                                    (results_df["metric"] == metric)]
            
            if len(cell_data) > 0:
                adopters = int(cell_data["adopters"].iloc[0])
                tours = int(cell_data["tours"].iloc[0])
                adoption_pct = cell_data["adoption_pct"].iloc[0]
                
                # Determine text color based on cell value
                text_color = "white" if val > (vmax * 0.6) else GREY_DARK
                
                # Main value
                ax.text(j, i - 0.15, f"{val:.1f}", 
                       ha="center", va="center",
                       color=text_color, fontsize=13, fontweight="bold")
                
                # Adopter count
                ax.text(j, i + 0.15, f"({adopters:,} suppliers)", 
                       ha="center", va="center",
                       color=text_color, fontsize=8, style='italic', alpha=0.9)
                
                # Tour count
                ax.text(j, i + 0.30, f"{tours:,} tours", 
                       ha="center", va="center",
                       color=text_color, fontsize=7, style='italic', alpha=0.8)

# Grid
ax.set_xticks(np.arange(len(metric_names)) - 0.5, minor=True)
ax.set_yticks(np.arange(len(segments)) - 0.5, minor=True)
ax.grid(which="minor", color="white", linestyle='-', linewidth=3)
ax.tick_params(which="minor", size=0)

# Colorbar
cbar = plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
cbar.set_label("Average Items Per Tour (among adopters)", fontsize=12, fontweight='bold')

# Explanation footer
explanation = (
    "Reading: The number shows average items per tour among tours from adopting suppliers.\n"
    "Example: If Individual pricing with >1 ticket category shows '3.2', it means tours from suppliers "
    "using multiple categories average 3.2 ticket categories per tour.\n"
    "Calculation: (1) Identify suppliers who use the feature, (2) Filter to tours from those suppliers, "
    "(3) Count items per tour, (4) Average across all tours."
)
ax.text(0.5, -0.15, explanation, 
        transform=ax.transAxes,
        fontsize=9,
        ha='center',
        bbox=dict(boxstyle="round,pad=0.8", facecolor=GREY_PALE, alpha=0.8, edgecolor=GREY))

plt.tight_layout()
plt.show()

print("\n✓ Chart: Configuration depth heatmap")

# =============================================================================
# KEY INSIGHTS: SHALLOW VS DEEP ENGAGEMENT
# =============================================================================

print("\n" + "="*100)
print("KEY INSIGHTS: SHALLOW VS DEEP CONFIGURATION")
print("="*100)

for metric_name in metric_names:
    metric_results = results_df[results_df["metric"] == metric_name]
    with_data = metric_results[metric_results["adopters"] > 0].copy()
    
    if len(with_data) > 0:
        # Weight by number of tours for overall average
        total_tours = with_data["tours"].sum()
        weighted_mean = (with_data["mean"] * with_data["tours"]).sum() / total_tours if total_tours > 0 else 0
        total_adopters = with_data["adopters"].sum()
        
        highest = with_data.loc[with_data["mean"].idxmax()]
        lowest = with_data.loc[with_data["mean"].idxmin()]
        
        # Classify depth
        if weighted_mean < 2:
            depth_level = "SHALLOW"
            interpretation = "Minimal configuration"
        elif weighted_mean < 4:
            depth_level = "MODERATE"
            interpretation = "Some real configuration"
        else:
            depth_level = "DEEP"
            interpretation = "Substantial configuration"
        
        print(f"\n{metric_name}:")
        print(f"  Depth level: {depth_level}")
        print(f"  Platform average: {weighted_mean:.1f} items per tour")
        print(f"  Total adopters: {int(total_adopters):,} suppliers across {int(total_tours):,} tours")
        print(f"  Interpretation: {interpretation}")
        print(f"  Deepest: {highest['segment']} ({highest['mean']:.1f} per tour, {int(highest['adopters']):,} suppliers)")
        print(f"  Shallowest: {lowest['segment']} ({lowest['mean']:.1f} per tour, {int(lowest['adopters']):,} suppliers)")
    else:
        print(f"\n{metric_name}:")
        print(f"  No adopters among top 80% GMV suppliers")

print("\n" + "="*100)
print("✓ Analysis complete: Configuration depth among adopters")
print("="*100)
